<a href="https://colab.research.google.com/github/rydraceo/RydraLLM/blob/main/Rydra_Portkey_PythonSDK_Test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install portkey-ai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 46.9 MB/s eta 0:00:00


In [ ]:
from portkey_ai import Portkey

# Your keys — keep this notebook private, never share the link
PORTKEY_API_KEY = "pk-paste-your-key-here"
OPENAI_VIRTUAL_KEY = "paste-your-virtual-key-here"

# Create the Portkey client
client = Portkey(
    api_key="gsd2b694oRQA3BLovNkCSYZ1cbP1",
    virtual_key="gpt-4o-mini"
)

# Make your first call
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "user",
            "content": "Say hello and tell me you are working correctly."
        }
    ]
)

# Print the response
print(response.choices[0].message.content)

Hello! I'm here and working correctly. How can I assist you today?


In [ ]:
# Fallback config — if OpenAI fails, automatically use Claude Haiku
import json

fallback_config = {
    "strategy": {
        "mode": "fallback"
    },
    "targets": [
        {
            "virtual_key": "gpt-4o-mini"
        },
        {
            "virtual_key": "claude-haiku-secondary-key"
        }
    ]
}

# Create client with fallback
client_with_fallback = Portkey(
    api_key="gsd2b694oRQA3BLovNkCSYZ1cbP1",
    config=json.dumps(fallback_config)
)

response = client_with_fallback.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "user",
            "content": "What model are you and are you responding correctly?"
        }
    ]
)

print(response.choices[0].message.content)

I am based on OpenAI's GPT-3 model, designed to understand and generate human-like text based on the input I receive. I aim to provide accurate and helpful responses based on the information I have been trained on, which includes a wide range of topics up until October 2021. If you have specific questions or concerns about my responses, please feel free to share, and I'll do my best to assist you!


In [ ]:
# Simulate what your daily cron job will send to Portkey
# This is fake data for testing — replace with real SQL output later

fake_venue_data = """
TODAY: Tuesday 22 April 2026

REVENUE
Today so far: $340.00
7-day daily average: $890.00

TOP SELLERS (this week vs last week)
Flat White: 47 orders (was 52)
Smashed Avo: 31 orders (was 28)
Eggs Benedict: 18 orders (was 24)
Truffle Fries: 6 orders (was 19)
Long Black: 41 orders (was 40)

DECLINING ITEMS (>30% drop)
Truffle Fries: 6 orders this week vs 19 last week (-68.4%)
Eggs Benedict: 18 orders this week vs 24 last week (-25.0%)

HOURLY PATTERN (avg orders/hour)
7:00 - 4.2 orders
8:00 - 12.1 orders
9:00 - 18.4 orders
10:00 - 9.2 orders
11:00 - 3.1 orders
12:00 - 8.7 orders
13:00 - 11.2 orders
14:00 - 2.3 orders
15:00 - 1.1 orders

BUNDLE OPPORTUNITIES (frequently ordered together)
Flat White + Smashed Avo: 28 times
Long Black + Eggs Benedict: 14 times
Flat White + Truffle Fries: 9 times
"""

system_prompt = """You are a sharp, data-driven hospitality business advisor for Australian cafes.

You receive structured sales data and return 3-5 specific, actionable business insights in JSON format.

RULES:
- Every insight must reference exact numbers from the data provided
- Actions must be implementable TODAY by a single staff member
- Estimate revenue impact in AUD
- Use Australian English. Currency in AUD.
- No generic advice. Reference specific items and numbers.

Return only valid JSON in this format:
{
  "insights": [
    {
      "type": "discount or bundle or timing or upsell or stock",
      "headline": "max 80 chars",
      "action": "max 200 chars of exactly what to do",
      "revenue_impact": "estimated AUD impact",
      "urgency": "today or this_week or monitor",
      "confidence": 0.0 to 1.0
    }
  ],
  "summary": "one sentence summary of biggest opportunity today"
}"""

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": fake_venue_data}
    ],
    response_format={"type": "json_object"},
    temperature=0.3
)

import json
result = json.loads(response.choices[0].message.content)

# Print it nicely
print("=== DAILY INSIGHTS FOR ROSA'S ===\n")
print(f"Summary: {result['summary']}\n")
for i, insight in enumerate(result['insights'], 1):
    print(f"Insight {i}: {insight['headline']}")
    print(f"  Action: {insight['action']}")
    print(f"  Revenue impact: {insight['revenue_impact']}")
    print(f"  Urgency: {insight['urgency']}")
    print(f"  Confidence: {insight['confidence']}")
    print()

=== DAILY INSIGHTS FOR ROSA'S ===

Summary: The biggest opportunity today is to bundle Flat White and Smashed Avo for increased sales.

Insight 1: Introduce a Flat White and Smashed Avo bundle
  Action: Offer a discount on a Flat White + Smashed Avo combo for $15.00 today.
  Revenue impact: Estimated increase of $210.00 if 14 bundles sold.
  Urgency: today
  Confidence: 0.8

Insight 2: Discount on Truffle Fries to boost sales
  Action: Reduce Truffle Fries price to $6.00 to encourage orders and clear stock.
  Revenue impact: Estimated increase of $36.00 if 6 orders sold.
  Urgency: today
  Confidence: 0.7

Insight 3: Upsell Long Black with Eggs Benedict
  Action: Suggest adding a Long Black for an extra $3.00 with each Eggs Benedict order.
  Revenue impact: Estimated increase of $42.00 if 14 upsells occur.
  Urgency: today
  Confidence: 0.75

Insight 4: Maximise orders during peak hours
  Action: Ensure staff are fully prepared and focused during 9:00-10:00 AM rush.
  Revenue impact: P